# Task 1 - Olist PostgreSQL Verification

**Author:** Rand Salem  
**Program:** Qafza Tech MLOps Training 2026/2027

This notebook connects to the local PostgreSQL database, checks the nine tables, runs joins, and demonstrates the future late-delivery target. It intentionally avoids full EDA because that is outside Task 1.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

env_path = Path('.env') if Path('.env').exists() else Path('../.env')
load_dotenv(env_path)
db_url = (
    f"postgresql+psycopg://{os.getenv('POSTGRES_USER', 'olist_user')}:"
    f"{os.getenv('POSTGRES_PASSWORD', '')}@"
    f"{os.getenv('POSTGRES_HOST', 'localhost')}:"
    f"{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'olist')}"
)
engine = create_engine(db_url)
print('Connected to PostgreSQL successfully.')

## 1. Verify table row counts

In [ ]:
tables = [
    'customers', 'geolocation', 'order_items', 'order_payments',
    'order_reviews', 'orders', 'products', 'sellers',
    'product_category_translation'
]
counts = []
for table in tables:
    count = pd.read_sql(f'SELECT COUNT(*) AS rows FROM {table}', engine).loc[0, 'rows']
    counts.append({'table': table, 'rows': count})
pd.DataFrame(counts).sort_values('table').reset_index(drop=True)

## 2. Join orders with customers

In [ ]:
orders_customers = pd.read_sql('''
SELECT o.order_id, o.order_status, o.order_purchase_timestamp,
       c.customer_city, c.customer_state
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_purchase_timestamp DESC
LIMIT 10
''', engine)
orders_customers

## 3. Multi-table join

In [ ]:
item_details = pd.read_sql('''
SELECT oi.order_id, oi.order_item_id, oi.price,
       COALESCE(t.product_category_name_english, p.product_category_name) AS category,
       s.seller_city, s.seller_state
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
JOIN sellers s ON s.seller_id = oi.seller_id
LEFT JOIN product_category_translation t
       ON t.product_category_name = p.product_category_name
LIMIT 10
''', engine)
item_details

## 4. Future ML target

`order_delivered_customer_date` creates the label but must not be used as an input feature, because it is available only after delivery and would cause target leakage.

In [ ]:
target_counts = pd.read_sql('''
SELECT CASE
         WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 'Late'
         ELSE 'On time'
       END AS delivery_class,
       COUNT(*) AS orders
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
GROUP BY delivery_class
ORDER BY delivery_class
''', engine)
target_counts